# 05 - GRU4Rec

GRU4Rec compresses the whole trip prefix into a single recurrent hidden state and scores every city from that one state, instead of pairwise transitions (Markov/FPMC) or attending over the whole sequence (SASRec):

$$h_t = \text{GRU}(h_{t-1}, \, \text{emb}(c_t)) \qquad \text{score}(i) = h_T \cdot \text{emb}(i)$$

First model in the project with genuine **long-range sequence memory** (Popularity -> Markov -> FPMC -> **GRU4Rec** -> SASRec).

### Benchmark protocol (same as `models_benchmarking.ipynb`)
1. **Tune** with seed 42: train on Train, select on validation **NDCG@10** -- 25 Optuna TPE trials (median pruner), full-catalogue cross-entropy, AdamW.
2. **Record $E_{best}$**, the peak epoch of the winning trial.
3. **Merge** Train + Validation.
4. **Train 5 fresh models** (seeds `[1, 7, 13, 2024, 99]`) for exactly $E_{best}$ epochs, no early stopping.
5. **Score on Test**; log **mean** and **std** across seeds to `results.csv`.

This is done twice -- **without** and **with** side features (`device_class`, `booker_country`, `affiliate_id`, summed into the input embedding) -- each variant with its own search and the identical budget. That is the ablation the plan requires.

All logic is in `src/models/gru4rec.py`. Running this notebook top to bottom does everything and ends by displaying `results.csv`.

> **Runtime:** 2 variants x 25 trials, then 2 x 5 final fits. Use a GPU. To smoke-test the pipeline first, set `N_TRIALS = 2` and `SEEDS = SEEDS[:2]` in the config cell below -- and don't trust those numbers.

In [ ]:
import torch

import os, sys, json
def find_root(start=None):
    d = os.path.abspath(start or os.getcwd())
    for _ in range(6):
        if os.path.exists(os.path.join(d, 'src', 'preprocessing.py')):
            return d
        d = os.path.dirname(d)
    return os.path.abspath(os.path.join(os.getcwd(), '..'))
ROOT = find_root(); sys.path.insert(0, ROOT)
import pandas as pd
from src.preprocessing import load_processed, preprocess
from src.evaluate import append_result
try:
    data = load_processed(os.path.join(ROOT, 'data', 'processed'))
except FileNotFoundError:
    data = preprocess(data_dir=os.path.join(ROOT, 'data'), sample_n_trips=5000)
print('loaded:', data['meta'])
if data['meta'].get('sample_n_trips') is not None:
    print('WARNING: processed.pkl was built on a subsample (sample_n_trips=%r). '
          'Delete data/processed/processed.pkl and rebuild with sample_n_trips=None '
          'before trusting or logging any numbers.' % data['meta']['sample_n_trips'])

RESULTS = os.path.join(ROOT, 'results.csv')
TUNING_DIR = os.path.join(ROOT, 'results', 'tuning'); os.makedirs(TUNING_DIR, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)

In [ ]:
from src.models.gru4rec import (tune_gru4rec, final_gru4rec, SIDE_FEATURES,
                                N_TRIALS, FINAL_SEEDS, MAX_EPOCHS, PATIENCE)
SEEDS = list(FINAL_SEEDS)
# N_TRIALS = 2; SEEDS = SEEDS[:2]        # <- uncomment for a quick smoke test only
print(f"trials/variant={N_TRIALS} | max_epochs={MAX_EPOCHS} | patience={PATIENCE} | seeds={SEEDS}")

## Run the protocol for both variants

For each variant: **tune** (validation only) -> read $E_{best}$ -> **final fits** on Train + Val for 5 seeds -> append the mean row and the std row to `results.csv`, and save the per-seed table and chosen hyper-parameters under `results/`.

Rows written: `GRU4Rec-no-side`, `GRU4Rec-no-side (std)`, `GRU4Rec-side`, `GRU4Rec-side (std)`.

### No-Side

In [ ]:
summaries = {}
for side_feats, tag in [((), "no-side")]:
    print(f"\n==================== GRU4Rec ({tag}) ====================")

    # steps 1-2: search on validation, record E_best
    study = tune_gru4rec(data, side_feats=side_feats, n_trials=N_TRIALS, device=DEVICE, show_progress_bar=False)
    best_params = dict(study.best_params)
    e_best = study.best_trial.user_attrs["best_epoch"]
    with open(os.path.join(TUNING_DIR, f"gru4rec-{tag}.json"), "w") as f:
        json.dump({"best_params": best_params, "E_best": e_best,
                   "best_val_NDCG@10": study.best_value, "n_trials": len(study.trials)}, f, indent=2)

    # steps 3-5: train+val, 5 seeds x E_best epochs, score on test
    per_seed, _ = final_gru4rec(data, best_params, epochs=e_best, side_feats=side_feats,
                                seeds=SEEDS, device=DEVICE)
    per_seed.to_csv(os.path.join(TUNING_DIR, f"gru4rec-{tag}-per-seed.csv"))

    mean, std = per_seed.mean(), per_seed.std()
    mean_row = {k: (int(v) if k == "n_params" else float(v)) for k, v in mean.items()}
    std_row = {k: float(v) for k, v in std.items()}
    append_result(RESULTS, f"GRU4Rec-{tag}", mean_row)
    append_result(RESULTS, f"GRU4Rec-{tag} (std)", std_row)
    summaries[tag] = {"best_params": best_params, "E_best": e_best,
                      "val_NDCG@10": round(study.best_value, 4),
                      "test_NDCG@10": f"{mean['NDCG@10']:.4f} +/- {std['NDCG@10']:.4f}"}

### Side

In [ ]:
summaries = {}
for side_feats, tag in [(SIDE_FEATURES, "side")]:
    print(f"\n==================== GRU4Rec ({tag}) ====================")

    # steps 1-2: search on validation, record E_best
    study = tune_gru4rec(data, side_feats=side_feats, n_trials=N_TRIALS, device=DEVICE, show_progress_bar=False)
    best_params = dict(study.best_params)
    e_best = study.best_trial.user_attrs["best_epoch"]
    with open(os.path.join(TUNING_DIR, f"gru4rec-{tag}.json"), "w") as f:
        json.dump({"best_params": best_params, "E_best": e_best,
                   "best_val_NDCG@10": study.best_value, "n_trials": len(study.trials)}, f, indent=2)

    # steps 3-5: train+val, 5 seeds x E_best epochs, score on test
    per_seed, _ = final_gru4rec(data, best_params, epochs=e_best, side_feats=side_feats,
                                seeds=SEEDS, device=DEVICE)
    per_seed.to_csv(os.path.join(TUNING_DIR, f"gru4rec-{tag}-per-seed.csv"))

    mean, std = per_seed.mean(), per_seed.std()
    mean_row = {k: (int(v) if k == "n_params" else float(v)) for k, v in mean.items()}
    std_row = {k: float(v) for k, v in std.items()}
    append_result(RESULTS, f"GRU4Rec-{tag}", mean_row)
    append_result(RESULTS, f"GRU4Rec-{tag} (std)", std_row)
    summaries[tag] = {"best_params": best_params, "E_best": e_best,
                      "val_NDCG@10": round(study.best_value, 4),
                      "test_NDCG@10": f"{mean['NDCG@10']:.4f} +/- {std['NDCG@10']:.4f}"}

## Selected configurations

In [ ]:
display(pd.DataFrame(summaries).T)

## Final results (`results.csv`)

In [ ]:
display(pd.read_csv(RESULTS))

## What to write up

- GRU4Rec should beat Markov wherever a trip's *earlier* stops (not just the last one) carry predictive signal -- check the trip-length slicing in `06_comparison.ipynb` for direct evidence.
- Report the side-feature ablation next to SASRec's: compare `GRU4Rec-side` vs `GRU4Rec-no-side` as mean +/- std over 5 seeds. A gap smaller than the std is not a real effect. If trip-level context hurts or doesn't help both neural models, that is a finding about the dataset, not an architecture quirk.
- Compute: see `n_params` / `train_secs` in `results.csv` for the accuracy-vs-cost discussion.
- The loss changed from sampled-negative BPR to full-catalogue cross-entropy and the optimiser to AdamW to match the benchmark protocol; SASRec/FPMC must use the same for the comparison to be fair.